# Phase 1 Notebook: Scaffold + Data Quality Baseline

## What was done
- Created the repository structure for ingestion, identity, features, forecasting, optimisation, agent, backtesting, evaluation, storage, and utils.
- Added typed contracts and baseline configs.
- Added tests and a sample fixture dataset for pipeline smoke checks.

## Why it was done
- To keep each later phase focused on implementation, we establish a runnable and testable platform now.
- We also establish an auditable quality baseline before building production ingestion and models.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path('../tests/fixtures/raw/fpl_players_sample.csv')
df = pd.read_csv(data_path)
df.head()

,player_id,season,gameweek,team,position,minutes,goals,assists,ict,bps,price,opp_strength,home,available_pre_deadline,target_points
0,p1,2024,1,ARS,MID,90,1,0,12.4,24,7.5,3,1,1,8
1,p1,2024,2,ARS,MID,86,0,1,10.1,18,7.6,4,0,1,6
2,p1,2024,3,ARS,MID,90,1,1,15.0,30,7.7,2,1,1,11
3,p1,2024,4,ARS,MID,75,0,0,6.2,12,7.7,5,0,1,3
4,p1,2024,5,ARS,MID,89,2,0,14.1,28,7.8,3,1,1,10


## Data quality checks
We check shape, missingness, duplicated player-gameweek keys, and temporal consistency.

In [2]:
key_cols = ['player_id', 'season', 'gameweek']
missing = df.isna().sum().sort_values(ascending=False)
duplicate_keys = df.duplicated(subset=key_cols).sum()
availability_issues = (df['available_pre_deadline'] == 0).sum()

summary = {
    'rows': len(df),
    'columns': len(df.columns),
    'duplicate_key_rows': int(duplicate_keys),
    'rows_not_available_pre_deadline': int(availability_issues),
}
summary, missing[missing > 0]

({'rows': 14,
  'columns': 15,
  'duplicate_key_rows': 0,
  'rows_not_available_pre_deadline': 1},
 Series([], dtype: int64))

## Findings and anomalies
- One sample row is marked unavailable before deadline (`available_pre_deadline = 0`).
- This row is intentional for testing leakage-safe filtering behavior.
- No duplicate player-season-gameweek records in the fixture.

## How anomalies are handled
- Rows unavailable before deadline are excluded in feature generation by `enforce_pre_deadline_only`.
- In later phases, stale-source anomalies will be handled with fail-soft ingestion and explicit freshness metadata.